# GeoAI Change Detection - Fast Version

Demonstrates STAC querying, lazy COG loading, NDVI change detection, and GeoJSON export.
Optimized for speed in Google Colab.

In [ ]:
!pip install -q pystac-client rasterio rioxarray geopandas matplotlib shapely

In [ ]:
import pystac_client
import rioxarray as rxr
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
from shapely.geometry import shape
import json
import rasterio
from rasterio.features import shapes

print("✅ Libraries installed and imported")

In [ ]:
# Small test AOI (change if you want)
bbox = [-100.0, 35.0, -99.5, 35.5]   # [min_lon, min_lat, max_lon, max_lat]

# Very short time windows for speed
time_before = "2023-06-15/2023-06-20"
time_after  = "2023-09-15/2023-09-20"

In [ ]:
# Use AWS Earth Search (often faster and more reliable in Colab)
catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")

print("Searching for scenes...")

search_before = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_before,
    query={"eo:cloud_cover": {"lt": 50}},
    max_items=2
)

search_after = catalog.search(
    collections=["sentinel-2-l2a"],
    bbox=bbox,
    datetime=time_after,
    query={"eo:cloud_cover": {"lt": 50}},
    max_items=2
)

items_before = list(search_before.items())
items_after = list(search_after.items())

print(f"Found {len(items_before)} scenes before and {len(items_after)} scenes after")

In [ ]:
def load_band(item, band_name):
    """Load one band lazily"""
    href = item.assets[band_name].href
    print(f"Loading {band_name} from {href[:80]}...")
    return rxr.open_rasterio(href, masked=True).squeeze()

if len(items_before) > 0 and len(items_after) > 0:
    red_b = load_band(items_before[0], "red")
    nir_b = load_band(items_before[0], "nir")
    red_a = load_band(items_after[0], "red")
    nir_a = load_band(items_after[0], "nir")

    ndvi_before = (nir_b - red_b) / (nir_b + red_b + 1e-6)
    ndvi_after  = (nir_a - red_a) / (nir_a + red_a + 1e-6)
    ndvi_diff   = ndvi_after - ndvi_before

    print("✅ NDVI difference calculated successfully")
else:
    print("No scenes found. Try expanding the date range or changing bbox.")

In [ ]:
# Visualize NDVI difference (optional)
if 'ndvi_diff' in locals():
    plt.figure(figsize=(8, 6))
    ndvi_diff.plot(cmap="BrBG", vmin=-0.5, vmax=0.5)
    plt.title("NDVI Change (After - Before)")
    plt.show()

In [ ]:
# === Export Change Polygons as GeoJSON ===
if 'ndvi_diff' in locals():
    threshold = 0.25
    change_mask = np.abs(ndvi_diff) > threshold

    # Use transform from one of the scenes
    with rasterio.open(items_after[0].assets["red"].href) as src:
        transform = src.transform
        crs = src.crs

    shapes_gen = shapes(change_mask.astype(np.uint8), transform=transform)

    features = []
    for geom, value in shapes_gen:
        if value == 1:
            features.append({
                "type": "Feature",
                "geometry": geom,
                "properties": {
                    "change_type": "Significant Change",
                    "ndvi_diff_mean": float(ndvi_diff.where(change_mask).mean().values)
                }
            })

    geojson_data = {"type": "FeatureCollection", "features": features}

    with open("change_polygons.geojson", "w") as f:
        json.dump(geojson_data, f, indent=2)

    print(f"✅ Successfully exported {len(features)} change polygons to change_polygons.geojson")
    print("Download this file from the left sidebar (Files) and upload it to the ROOT of your GitHub repo.")

## Key Achievements
- Fast STAC search using AWS Earth Search
- Lazy loading of Sentinel-2 COGs (no full download)
- NDVI-based automated change detection
- Vector polygons exported as GeoJSON ready for the web map